# Introduction to python for hydrologists &mdash; pandas

<img src = "./data/pandas/panda.jpg">

## Not that type of Panda -- Python's Pandas package

Pandas is a powerful, flexible and easy to use open source data analysis and manipulation tool. Pandas is commonly used for operations that would normally be done in a spreadsheet environment and includes powerful data analysis and manipulation tools.

Let's begin by importing the libraries and setting our data path 

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
from dataretrieval import nwis
import statsmodels.api as sm
from matplotlib.backends.backend_pdf import PdfPages
from scipy.signal import detrend

data_path = Path("data", "pandas")

## Get site information data for part of the Russian River near Guerneville, CA from NWIS

<img src = "./data/pandas/stock_russian.jpg">

In [ ]:
bbox = [-123.10, 38.45, -122.90, 38.55]

There are a lot of sites in this location, let's use the NWIS data retrevial tool to get info on all of them and then use this data to learn about pandas

In [ ]:
info, metadata = nwis.get_info(bBox=[str(i) for i in bbox])
if 'geometry' in list(info):
    # dataretreival now returns as geopandas dataframe, drop "geometry" column for this example
    info.drop(columns=["geometry"], inplace=True)

In [ ]:
info.to_csv(data_path / "site_info.csv", index=False)

The data returned to us from our NWIS query `info` was returned to us as pandas `DataFrame`. We'll be working with this to start learning about the basics of pandas.

## Viewing data in pandas

Pandas has built in methods to inspect `DataFrame` objects. We'll look at a few handy methods:

   - `.head()`: inspect the first few rows of data
   - `.tail()`: inspect the last few rows of data
   - `.index()`: show the row indexes
   - `.columns()`: show the column names
   - `.describe()`: statistically describe the data

In [ ]:
info.head()

In [ ]:
info.tail()

In [ ]:
# print the index names
print(info.index)

In [ ]:
# print the column names
print(info.columns)

# print the column names as a list
print(list(info))

The `describe()` method is only useful for numerical data. This example is does not have a lot of useful data for `describe()`, however we'll use it again later

In [ ]:
info.describe()

## Getting data from a pandas dataframe

There are multiple methods to get data out of a pandas dataframe as either a "series", numpy array, or a list

Let's start by getting data as a series using a few methods

In [ ]:
# get a series of site numbers by key
info["site_no"]

In [ ]:
# get station names by attribute
info.station_nm

getting data from a dataframe as a numpy array can be accomplished by using `.values`

In [ ]:
info.station_nm.values

### Selection by position

We can get data by position in the dataframe using the `.iloc` attribute 

In [ ]:
info.iloc[0:2, 1:4]

### Selection by label

Pandas allows the user to get data from the dataframe by index and column labels

In [ ]:
info.loc[1, ["site_no", "station_nm", "site_tp_cd"]]

### Boolean indexing

pandas dataframes supports boolean indexing that allows a user to create a new dataframe with only the data that meets a boolean condition defined by the user.

Let's get a dataframe of only groundwater sites from the `info` dataframe

In [ ]:
dfgw = info[info["site_tp_cd"] == "GW"]
dfgw

We can also reset the column indexing using `set_index()`

In [ ]:
dfsite = dfgw.set_index("site_no")
dfsite.head()

And create a pivot table using the `.pivot()` method. For this example we'll pivot the `info` dataframe using the site type code as an index, site number as columns, and query the project associated with it.

In [ ]:
dfsite = info.pivot(index="site_tp_cd", columns=["site_no"], values="project_no")
dfsite.head()

### Reading and writing data to `.csv` files

Pandas has support to both read and write many types of files. For this example we are focusing on `.csv` files. For information on other file types that are supported see the [ten minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html#importing-and-exporting-data) tutorial documentation

For this part we'll write a new `.csv` file of the groundwater sites that we found in NWIS using `to_csv()`.

`to_csv()` has a bunch of handy options for writing to file. For this example, I'm going to drop the index column while writing by passing `index=False`

In [ ]:
csv_file = data_path / "RussianRiverGWsites.csv"
dfgw.to_csv(csv_file, index=False)

Now we can load the csv file back into a pandas dataframe with the `read_csv()` method.

In [ ]:
dfgw2 = pd.read_csv(csv_file)
dfgw2

## Class exercise 1 

Using the methods presented in this notebook create a `DataFrame` of surface water sites from the `info` dataframe, write it to a csv file named `"RussianRiverSWsites.csv"`, and read it back in as a new `DataFrame`

## Get timeseries data from NWIS for surface water sites

Now to start working with stream gage data from NWIS. We're going to send all surface water sites to NWIS and get only sites with daily discharge measurements.

([11467000](https://waterdata.usgs.gov/nwis/inventory?agency_code=USGS&site_no=11467000))

In [ ]:
# filter for only SW sites
dfsw = info[info.site_tp_cd == "ST"]

# create query
sites = dfsw.site_no.to_list()
pcode = "00060"
start_date = "1939-09-22"
end_date = "2023-12-31"
df, metadata = nwis.get_dv(
    sites=sites, 
    start=start_date, 
    end=end_date, 
    parameterCd=pcode, 
    multi_index=False
)
df.to_csv(data_path / "RR_gage_data.csv")

In [ ]:
df = pd.read_csv(data_path / "RR_gage_data.csv", dtype={'site_no': object})
df["datetime"] = pd.to_datetime(df["datetime"])
df = df.set_index("datetime")
df.head()

In [ ]:
unique_sites = df.site_no.unique()
unique_sites

Awesome! There are two gages in this study area that have daily discharge data!

Let's do some manipulation to the data frame so we can plot both of the gages

In [ ]:
tdf = df[df.site_no == unique_sites[-1]]
dfx = df[df.site_no == unique_sites[0]]

In [ ]:
dfm = pd.merge(dfx, tdf, how="outer", left_index=True, right_index=True)
dfm

### Inspect the data using `.plot()`

In [ ]:
ax = dfm.plot()
ax.set_ylabel(r"cubic feet per second")
ax.set_yscale("log");

### Dropping columns

We can remove columns of data from the dataframe using pandas built in `drop()` method

In [ ]:
dfm = dfm.drop(columns=['00060_Mean_cd_x', '00060_Mean_cd_y', 'site_no_x', 'site_no_y'])
dfm.head()

### Updating column names

Let's update column names for these gages based on their locations instead of their USGS gage id. First we'll get name information from the `info` class and then we'll use the station name information to remap the column names.

the `rename()` method accepts a dictionary that is formatted `{current_col_name: new_col_name}`

In [ ]:
info.head()

In [ ]:
remap = {}
for ix, i in enumerate(list(dfm)):
    site_no = unique_sites[ix]
    desc = info[info.site_no == site_no]["station_nm"].values[0]
    colname = "_".join(desc.lower().split()[:2])
    remap[i] = colname
    
df2 = dfm.rename(columns=remap)
ax = df2.plot()
ax.set_yscale("log")

### Adding a new column of data to the dataframe

Adding new columns of data is relatively simple in pandas. The call signature is similar to a dictionary where df["key"] = "some_array_of_values"

Let's add a year column (water year) to the dataframe by getting the year from the index and adjusting it using `np.where()`

In [ ]:
df2['year'] = np.where(df2.index.month >=10, df2.index.year + 1, df2.index.year)

### Manipulating data

Columns in a pandas dataframe can be manipulated similarly to working with a dictionary of numpy arrays. The discharge data units are $\frac{ft^3}{s}$, let's accumulate this to daily discharge in $\frac {ft^3}{d}$.

The conversion for this is:

## $ \frac{1 ft^3}{s} \times \frac{60s}{min} \times \frac{60min}{hour} \times \frac{24hours}{day} \rightarrow \frac{ft^3}{day} $

and then convert that to acre-feet (1 acre-ft = 43559.9)

In [ ]:
conv = 60 * 60 * 24
df2["russian_r_cfd"] = df2.russian_r * conv
df2["russian_r_af"] = df2.russian_r_cfd / 43559.9
df2

# now let's plot up discharge from 2018 - 2023
dfplt = df2[(df2["year"] >= 2018) & (df2["year"] <= 2023)]
ax = dfplt["russian_r_af"].plot()
ax.set_yscale("log")
ax.set_ylabel("Q, in acre-ft per day");


## Class exercise 2:

Using the methods described in the notebook. Convert the discharge for `"austin_c"` from $\frac{ft^3}{s}$ acre-ft by adding two additional fields to `df2` named `"austin_c_cfd"` and `"austin_c_af"`. 

After these two fields have been added to df2 use boolean indexing to create a new dataframe from 2015 through 2020. Finally plot the discharge data (in acre-ft) for "austin_cr".

**bonus exercise** try to plot both `"russian_r_af"` and `"austin_c_af"` on the same plot

### groupby: grouping data and performing mathematical operations on it

`groupby()` is a powerful method that allows for performing statistical operations over a groups of "common" data within a dataframe. 

For this example we'll use it to get mean daily flows for the watershed. Pandas will group all common days of the year with each other and then calculate the mean value of these via the function `.mean()`. `groupby()` also supports other operations such as `.median()`, `.sum()`, `max()`, `min()`, `std()`, and other functions. 

In [ ]:
df2["day_of_year"] = df2.index.day_of_year
df2["austin_c_cfd"] = df2.austin_c * conv
df2["austin_c_af"] = df2.austin_c_cfd / 43559.9
df_mean_day = df2.groupby(by=["day_of_year"], as_index=False)[["russian_r_af", "austin_c_af"]].mean()
ax = df_mean_day[["russian_r_af", "austin_c_af"]].plot()
ax.set_yscale("log")

We can see that around March flow starts decreasing heavily and doesn't recover until sometime in october. What's going on here? Let's load some climate data and see what's happening.

In [ ]:
cimis_file = data_path / "santa_rosa_CIMIS_83.csv"
df_cimis = pd.read_csv(cimis_file)
drop_list = ["qc"] + [f"qc.{i}" for i in range(1, 22)]
df_cimis.drop(columns=drop_list, inplace=True)
df_cimis

In [ ]:
df_mean_clim = df_cimis.groupby(by=["Jul"], as_index=False)[["ETo (in)", "Precip (in)"]].mean()
df_mean_clim = df_mean_clim.rename(columns={"Jul": "day_of_year"})
df_mean_clim.head()

### Merging two `DataFrames` with an inner join

Inner joins can be made in pandas with the `merge()` method. An "inner join" joins two dataframes on a common key or values; when a key or value exists in one dataframe, but not the other, that row of data will be excluded from the joined dataframe. 

There are a number of other ways to join dataframes in pandas. Detailed examples and discussion of the different merging methods can be found [here](https://realpython.com/pandas-merge-join-and-concat/)

In [ ]:
df_merged = pd.merge(df_mean_day, df_mean_clim, how="inner", on=["day_of_year",])
df_merged.head()

In [ ]:
cols = [i for i in list(df_merged) if i != "day_of_year"]
ax = df_merged[cols].plot()
ax.set_yscale("log")

This starts to make a little more sense! We can see that this area gets most of it's precipitation in the winter and early spring. The evapotranspiration curve shows what we'd expect: that there is more evapotranspiration in the summer than the winter. 

## Stepping back for a minute: exploring Merge and Concat

Merge and concat are very powerful and important methods for combining pandas DataFrames (and later geopandas GeoDataFrames). This section takes a moment, as an aside to explore some of the common funtions and options for performing merges and concatenation. 


### The `merge` method

Pandas `merge` method allows the user to join two dataframes based on column values. There are a few methods that can be used to control the type of merge and the result.

   - `inner`: operates as a intersection of the data and keeps rows from both dataframes if the row has a matching key in both dataframes.
   - `outer`: operates as a union and joins all rows from both dataframes, fills columns with nan where data is not available for a row.
   - `left`: keeps all rows from the 'left' dataframe and joins columns from the right
   - `right`: keeps all rows from the 'right' dataframe and joins columns from the left

In [ ]:
ex0 = {
    "name": [], 
    "childhood_dream_job": []
}
ex1 = {
    "name": [],
    "age": []
}

### The `concat` method

Pandas `concat` method allows the user to concatenate dataframes together

In [ ]:
gw = info[info.site_tp_cd == "GW"]
sw = info[info.site_tp_cd == "ST"]
sw = sw[list(sw)[0:8]]
sw.head()

## Back to working with our climate and stream data

Is there anything else we can look at that might give us more insights into this basin? 

Let's try looking at long term, yearly trends to see what's there.

In [ ]:
year = []
for dstr in df_cimis.Date.values:
    mo, d, yr = [int(i) for i in dstr.split("/")]
    if mo < 10:
        year.append(yr)
    else:
        year.append(yr + 1)
df_cimis["year"] = year
df_cimis.head()

In [ ]:
df2.head()

### `.aggregate()`

The pandas `.aggregate()` can be used with `.groupy()` to perform multiple statistical operations.

Example usage could be:

```python
    
    agdf = df2.groupby(by=["day_of_year"], as_index=False)["austin_c_af"].aggregate(["min", "max", "mean", "std"])
```

And would return a dataframe grouped by the day of year and take the min, max, mean, and standard deviation of these data.

## Class exercise 3: groupy, aggregate, and merge

For this exercise we need to produce yearly mean and standard deviation flows for the `russian_r_af` variable and merge the data with yearly mean precipitation and evapotranspiration  

**Hints:**  
   - `df2` and `df_cimis` are the dataframes these operations should be performed on  
   - use `.groupby()` to group by the year and do `precip` and `ETo` in separate groupby routines
   - rename the columns in the ETo aggregated dataframe and the Precip dataframe to `mean_et`, `std_et`, `mean_precip`, `std_precip`
   - `mean` and `std` can be used to calculate mean and std flows  
   
Name your final joined dataframe `df_yearly`

Lets examine the long term flow record for the Russian River

In [ ]:
df_y_flow = df2.groupby(by=["year"], as_index=False)["russian_r_af"].aggregate(["mean", "std"])

fig = plt.figure(figsize=(12,4))

lower_ci = df_y_flow["mean"] - 2 * df_y_flow['std']
lower_ci = np.where(lower_ci < 0, 0, lower_ci)
upper_ci = df_y_flow["mean"] + 2 * df_y_flow['std']
ax = df_y_flow["mean"].plot(style="b.-")
ax.fill_between(df_y_flow.index, lower_ci, upper_ci, color="b", alpha=0.5)
ax.set_yscale("log");

From this plot it looks like the long term trend in yearly discharge in the Russian River has been decreasing. We will revisit and test this later. 

First let's see if there are any relationships between yearly discharge and climate

In [ ]:
dfg0 = df2.groupby(by=["year"], as_index=False)["russian_r_af"].aggregate(["mean", "std", "sum"])
dfg1 = df_cimis.groupby(by=["year"], as_index=False)["ETo (in)"].aggregate(["mean", "std", "sum"])
dfg2 = df_cimis.groupby(by=["year"], as_index=False)["Precip (in)"].aggregate(["mean", "std", "sum"])
dfm1 = pd.merge(dfg0, dfg1, on=["year"], suffixes=(None, "_et"))
df_yearly = pd.merge(dfm1, dfg2, on=["year"], suffixes=(None, "_precip"))
df_yearly.set_index("year", inplace=True)

fig = plt.figure(figsize=(12,4))

lower_ci = df_yearly["mean"] - 2 * df_yearly['std']
lower_ci = np.where(lower_ci < 0, 0, lower_ci)
upper_ci = df_yearly["mean"] + 2 * df_yearly['std']
ax = df_yearly["mean"].plot(style="b.-", label="flow_af")
ax.fill_between(df_yearly.index.values, lower_ci, upper_ci, color="b", alpha=0.5)
ax.set_yscale("log")
ax2 = ax.twinx()
ax2.plot(df_yearly.index.values, df_yearly.sum_precip.values, "k--", lw=1.5, label="Precip")
ax2.plot(df_yearly.index.values, df_yearly.sum_et.values, "r.--", lw=1.5, label="ET")
plt.legend()
plt.show();

As expected, precipitation amount is the main driver of the yearly discharge regime here

### Baseflow separation

Baseflow separation is a method to separate the quick response hydrograph (storm runoff) from the long term flow. We're going to go back to the complete daily dataset to perform this operation. The following cell contains a low pass filtration method that is used for digital baseflow separation. We'll use this method in our analysis

In [ ]:
def _baseflow_low_pass_filter(arr, beta, enforce):
    """
    Private method to apply digital baseflow separation filter
    (Lyne & Hollick, 1979; Nathan & Mcmahon, 1990;
    Boughton, 1993; Chapman & Maxwell, 1996).

    This method should not be called by the user!

    Method removes "spikes" which would be consistent with storm flow
    events from data.

    Parameters
    ----------
    arr : np.array
        streamflow or municipal pumping time series

    beta : float
        baseflow filtering parameter that ranges from 0 - 1
        values in 0.8 - 0.95 range used in literature for
        streamflow baseflow separation

    enforce : bool
        enforce physical constraint of baseflow less than measured flow

    Returns
    -------
        np.array of filtered data
    """
    # prepend 10 records to data for initial spin up
    # these records will be dropped before returning data to user
    qt = np.zeros((arr.size + 10,), dtype=float)
    qt[0:10] = arr[0:10]
    qt[10:] = arr[:]

    qdt = np.zeros((arr.size + 10,), dtype=float)
    qbf = np.zeros((arr.size + 10,), dtype=float)

    y = (1. + beta) / 2.

    for ix in range(qdt.size):
        if ix == 0:
            qbf[ix] = qt[ix]
            continue

        x = beta * qdt[ix - 1]
        z = qt[ix] - qt[ix - 1]
        qdt[ix] = x + (y * z)

        qb = qt[ix] - qdt[ix]
        if enforce:
            if qb > qt[ix]:
                qbf[ix] = qt[ix]
            else:
                qbf[ix] = qb

        else:
            qbf[ix] = qb

    return qbf[10:]


def baseflow_low_pass_filter(arr, beta=0.9, T=1, enforce=True):
    """
    User method to apply digital baseflow separation filter
    (Lyne & Hollick, 1979; Nathan & Mcmahon, 1990;
    Boughton, 1993; Chapman & Maxwell, 1996).

    Method removes "spikes" which would be consistent with storm flow
    events from data.

    Parameters
    ----------
    arr : np.array
        streamflow or municipal pumping time series

    beta : float
        baseflow filtering parameter that ranges from 0 - 1
        values in 0.8 - 0.95 range used in literature for
        streamflow baseflow separation

    T : int
        number of filtering passes to apply to the data

    enforce : bool
        enforce physical constraint of baseflow less than measured flow

    Returns
    -------
        np.array of baseflow
    """
    for _ in range(T):
        arr = _baseflow_low_pass_filter(arr, beta, enforce)

    return arr


## Class exercise 4: baseflow separation

Using the full dataframe discharge dataframe, `df2`, get the baseflows for the Russian River (in acre-ft) and add these to the dataframe as a new column named `russian_bf_af`.

The function `baseflow_low_pass_filter()` will be used to perform baseflow. This function takes a numpy array of stream discharge and runs a digital low pass filtration method to calculate baseflow. 

After performing baseflow separation plot both the baseflow and the total discharge for the years 2015 - 2017.

**Hints:**
   - make sure to use the `.values` property when you get discharge data from the pandas dataframe
   - feel free to play with the input parameters `beta` and `T`. 
      - `beta` is commonly between 0.8 - 0.95 for hydrologic problems.
      - `T` is the number of filter passes over the data, therefore increasing `T` will create a smoother profile. I recommend starting with a `T` value of around 5 and adjusting it to see how it changes the baseflow profile.

## Looking at trends and seasonal cycles in the data with `statsmodels`

`statsmodels` is a Python module that provides classes and functions for the estimation of many different statistical models, as well as for conducting statistical tests, and statistical data exploration. For more information on `statsmodels` and it's many features visit [the getting started guide](https://www.statsmodels.org/dev/gettingstarted.html)

For this example we'll be using time series analysis to perform a seasonal decomposition using moving averages. The `statsmodels` function `seasonal_decompose` will perform this analysis and return the data trend, seasonal fluctuations, and residuals.

In [ ]:
df2["russian_bf_af"] = baseflow_low_pass_filter(df2["russian_r_af"].values, beta=0.9, T=5)
# drop nan values prior to running decomposition
dfsm = df2[df2['russian_bf_af'].notna()]

dfsm.head()
# decompose
decomposition = sm.tsa.seasonal_decompose(dfsm["russian_bf_af"], period=365, model="additive")
 
decomposition.plot()
mpl.rcParams['figure.figsize'] = [14, 14];

Now let's plot our baseflow trend against the baseflow we calculated 

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

ax = df2["russian_r_af"].plot(ax=ax)
ax.plot(decomposition.trend.index.values, decomposition.trend.values, "r-", lw=1.5)
ax.set_yscale('log')
plt.savefig("russian_river.png")


What's going on at the end of the 1970's? Why is the baseflow trend so low and peak flows are also low? 

From ["California's Most Significant Droughts"](https://cawaterlibrary.net/wp-content/uploads/2017/05/CalSignficantDroughts_v10_int.pdf), Chapter 3 

The setting for the 1976-77 drought differed significantly from the dry times of the 1920s-30s. Although
only a two-year event, its hydrology was severe.
Based on 114 years of computed statewide runoff,
1977 occupies rank 114 (driest year) and 1976 is in
rank 104. The drought was notable for the impacts
experienced by water agencies that were unprepared
for such conditions.  One reason for the lack of
preparedness was the perception of relatively ample
water supplies in most areas of the state. 

We can also see the analysis that there is an overall reduction in baseflow from 1940's to the 2020's

### Writing yearly plots of data to a PDF document

We can write the plots we make with pandas to a multipage pdf using matplotlib's `PdfPages`. For this example we'll make yearly plots of all of the discharge, baseflow, and baseflow trend and bundle them into a single pdf document.

In [ ]:
# add the decomposition trend to the dataframe to make plotting easy
df2["baseflow_trend"] = decomposition.trend

pdf_file = data_path / "all_years_russian.pdf"
with PdfPages(pdf_file) as outpdf:
    for year in sorted(df2.year.unique()):
        plt.figure()
        tdf = df2[df2.year == year]
        tdf[["russian_r_af", "russian_bf_af", "baseflow_trend"]].plot()
        plt.xlabel("date")
        plt.ylabel("acre-feet")
        plt.ylim([1, 300000])
        plt.yscale("log")
        plt.tight_layout()
        outpdf.savefig()
        plt.close('all')

Go into the "data/pandas" directory and open up the file "all_years_russian.pdf" to check out the plots.

## Back to the basics:

Now that we've done some analysis on real data, let's go back to the basic and briefly discuss how to adjust data values in pandas


### Adjusting a whole column of data values

Because pandas series objects (columns) are built off of numpy, we can perform mathematical operations in place on a whole column similarly to numpy arrays


In [ ]:
df3 = df2.copy()
# Adding values
df3["russian_r"] += 10000

# Subtracting values
df3["russian_r"] -= 10000

# multiplication
df3["russian_r"] *= 5

# division
df3["russian_r"] /= 5

# more complex operations
df3["russian_r"] = np.log10(df3["russian_r"].values)
df3.head()

### updating values by position

We can update single or multiple values by their position in the dataframe using `.iloc`

In [ ]:
df3.iloc[0, 1] = 999
df3.head()

### updating values based on location

We can update values in the dataframe based on their index and column headers too using `.loc`

In [ ]:
df3.loc[df3.index[0], 'russian_r'] *= 100
df3.head()

## Class exercise 5: setting data

1) The russian river discharge in cubic feet per day highly variable and represented by large numbers. To make this data easier to read and plot, update it to a log10 representation.

2) Then use either a location or position based method to change the value in austin_c on 10-4-1939 to 0.

3) Finally display the first 5 records in the notebook.

## Now for some powerful analysis using fourier analysis to extract the data's signal.

Some background for your evening viewing: https://youtu.be/spUNpyF58BY

We have to detrend the data for fast fourier transforms to work properly. Here's a discussion on why: 

https://groups.google.com/forum/#!topic/comp.dsp/kYDZqetr_TU

Fortunately we can easily do this in python using scipy!

https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.detrend.html

Let's create a new dataframe with only the data we're interested in analysing.

In [ ]:
df_data = df2[["russian_r_af", "year"]]
df_data = df_data.rename(columns={"russian_r_af": "Q"})
df_data.describe()

#### Now we're going to detrend the data

But first let's also set the water year by date on this dataframe and drop nan values from the dataframe

In [ ]:
df_data['water_year'] = df_data.index.shift(30+31+31,freq='d').year
df_data.dropna(inplace=True)
df_data["detrended"] = detrend(df_data.Q.values)

In [ ]:
fig = plt.figure(figsize=(12,6))
ax0 = plt.subplot(3,1,1)
ax0.set_yscale("log")
plt.plot(df_data.index.values, df_data.detrended.values)
plt.title('detrended signal')

ax1 = plt.subplot(3,1,2)
ax1.set_yscale("log")
plt.plot(df_data.index.values, df_data.Q.values)
plt.title('Raw Signal')

plt.subplot(3,1,3)
plt.plot(df_data.index.values, df_data.Q.values - df_data.detrended.values)
plt.title('Difference')

plt.tight_layout()

### Evaluate and plot the Period Spectrum to see timing of recurrence

Here we've created a function that performs fast fourier transforms and then plot's the spectrum for signals of various lengths.

In [ ]:
def fft_and_plot(df, plot_dominant_periods=4):
    N = len(df)
    yf = np.fft.fft(df.detrended)
    yf = np.abs(yf[:int(N / 2)])
    
    # get the right frequency 
    # https://docs.scipy.org/doc/numpy/reference/generated/numpy.fft.fftfreq.html#numpy.fft.fftfreq
    d = 1. # day
    f = np.fft.fftfreq(N, d)[:int(N/2)]
    f[0] = .00001
    per = 1./f # days
    
    fig = plt.figure(figsize=(12,6))
    ax = plt.subplot(2,1,1)
    plt.plot(per, yf)
    plt.xscale('log')

    top=np.argsort(yf)[-plot_dominant_periods:]
    j=(10-plot_dominant_periods)/10
    for i in top:
        plt.plot([per[i], per[i]], [0,np.max(yf)], 'r:')
        plt.text(per[i], j*np.max(yf), f"{per[i] :.2f}")
        j+=0.1

    plt.title('Period Spectrum')
    plt.grid()
    ax.set_xlabel('Period (days)')
    plt.xlim([1, 1e4])

    plt.subplot(2,1,2)
    plt.plot(df.index.values, df.Q.values)
    plt.title('Raw Signal')
    plt.tight_layout()

### Now let's look at the whole signal

In [ ]:
fft_and_plot(df_data)

We see a strong annual frequency that corresponds to spring snowmelt from the surrounding mountain ranges. The second strongest frequency is biannually, and is likely due to the onset of fall and winter rains

### Okay, what about years before dams were installed on the Russian River.

Let's get in the wayback machine and look at only the years prior to 1954!

In [ ]:
fft_and_plot(df_data[df_data.index.year < 1954])

The binanual and annual signals have a much wider spread, which suggests that peak runoff was a little more variable in it's timing compared to the entire data record.

### What if we focus in on after 1954?

In [ ]:
fft_and_plot(df_data[(df_data.index.year > 1954)])

It looks like flows are more controlled with a tighter periods.

So post dam construction, flows are likely more controlled with regular seasonal release schedules. 

---

## 🤖 AI Sidebar: Data Manipulation with AI Assistance

*This sidebar demonstrates how an AI coding assistant approaches the same kind of pandas data manipulation tasks you practiced above — reading NWIS data, cleaning it, and computing summary statistics with groupby. All AI-generated code below was captured ahead of time; no live AI calls are made.*

**Time estimate:** ~5-8 minutes

### The Task

Throughout this notebook you worked with NWIS streamflow data for the Russian River: reading CSV files, filtering by site type, merging DataFrames, computing water-year columns, performing unit conversions, and using `groupby` to calculate monthly and yearly statistics.

Now let's see how an AI assistant handles a realistic pandas task: **reading a daily streamflow CSV, cleaning it, computing monthly mean discharge in acre-feet, and identifying the driest month on record.** This kind of data wrangling is routine in hydrology, so it's a good test of whether AI-generated pandas code can be trusted.

### AI Approach

**Prompt used:**

> *"I have a CSV file of daily USGS streamflow data with columns `datetime`, `site_no`, `00060_Mean` (discharge in cubic feet per second), and `00060_Mean_cd` (qualifier code). Read the file into a pandas DataFrame, parse the datetime column as the index, drop rows where discharge is NaN, add a water-year column (water year starts October 1), convert discharge from cfs to acre-feet per day, then compute monthly mean discharge in acre-feet per day grouped by year and month. Finally, find the month with the lowest mean discharge. Use only pandas and numpy."*

**AI-generated code:**

In [ ]:
# AI-GENERATED CODE (pre-captured)
import pandas as pd
import numpy as np

# Read the daily streamflow CSV
csv_path = data_path / "RR_gage_data.csv"
df_ai = pd.read_csv(csv_path, dtype={"site_no": str})
df_ai["datetime"] = pd.to_datetime(df_ai["datetime"])
df_ai = df_ai.set_index("datetime")

# Keep only the first site for a clean example
first_site = df_ai["site_no"].unique()[0]
df_ai = df_ai[df_ai["site_no"] == first_site].copy()

# Drop rows where discharge is NaN
df_ai = df_ai.dropna(subset=["00060_Mean"])

# Add water year column (Oct 1 starts the new water year)
df_ai["water_year"] = np.where(
    df_ai.index.month >= 10,
    df_ai.index.year + 1,
    df_ai.index.year,
)

# Convert cfs to acre-feet per day
# 1 cfs = 1.9835 acre-feet per day
df_ai["discharge_af"] = df_ai["00060_Mean"] * 1.9835

# Monthly mean discharge grouped by water year and month
df_ai["month"] = df_ai.index.month
monthly_mean = df_ai.groupby(
    ["water_year", "month"], as_index=False
)["discharge_af"].mean()

# Find the month with the lowest mean discharge
driest_row = monthly_mean.loc[monthly_mean["discharge_af"].idxmin()]
print(f"Driest month on record: water year {int(driest_row['water_year'])}, "
      f"month {int(driest_row['month'])} — "
      f"mean discharge {driest_row['discharge_af']:.2f} acre-ft/day")

monthly_mean.head(10)

Let's verify the AI-generated code against values we can compute manually:

In [ ]:
# Verification: check the cfs-to-acre-feet conversion factor
# 1 cfs = 1 ft^3/s * 86400 s/day / 43559.9 ft^3/acre-ft = 1.9835 acre-ft/day
manual_factor = (60 * 60 * 24) / 43559.9
ai_factor = 1.9835
print(f"Manual conversion factor: {manual_factor:.4f}")
print(f"AI conversion factor:     {ai_factor:.4f}")
print(f"Match: {abs(manual_factor - ai_factor) < 0.001}")
print()

# Verification: check water year assignment for a known date
# November 15, 2020 should be water year 2021
nov_rows = df_ai.loc["2020-11-15":"2020-11-15"]
if len(nov_rows) > 0:
    print(f"Nov 15, 2020 water year: {nov_rows['water_year'].values[0]}  (expected: 2021)")

# March 15, 2020 should be water year 2020
mar_rows = df_ai.loc["2020-03-15":"2020-03-15"]
if len(mar_rows) > 0:
    print(f"Mar 15, 2020 water year: {mar_rows['water_year'].values[0]}  (expected: 2020)")

### Comparison

| Aspect | Manual Approach | AI Approach |
|--------|----------------|-------------|
| **Time** | 10-15 min to write the full pipeline step by step | ~1 minute to generate from a detailed prompt |
| **Understanding** | You learn each pandas method (`groupby`, `dropna`, `set_index`) by using it | You get working code but may miss how the pieces connect |
| **Correctness** | You derive the conversion factor yourself (86400 / 43559.9) | AI used a rounded constant (1.9835) — correct but hides the derivation |
| **Index handling** | You learn about DatetimeIndex behavior through practice | AI sets the index correctly but doesn't explain *why* it matters |
| **Groupby logic** | You choose groupby keys based on your analysis question | AI picks reasonable keys, but may not match your exact intent |

### What Could Go Wrong

Pandas code from AI assistants often looks correct at first glance but can contain subtle data-handling errors. Here are two common failure modes:

**1. Index alignment errors during merge or arithmetic**

Pandas aligns operations on the index by default. If two DataFrames have different indexes (e.g., one is integer-indexed and the other has a DatetimeIndex), arithmetic or merge operations can silently produce NaN values or misaligned rows. AI-generated code frequently resets or sets indexes inconsistently across steps.

In [ ]:
# Demonstration: index alignment trap
# Suppose AI generates two series with different indexes and subtracts them
s1 = pd.Series([10, 20, 30], index=[0, 1, 2], name="flow_a")
s2 = pd.Series([5, 15, 25], index=[1, 2, 3], name="flow_b")

# AI might write: difference = s1 - s2
difference = s1 - s2
print("s1 - s2 with mismatched indexes:")
print(difference)
print("\n-> NaN values appear where indexes don't overlap!")
print("   In a streamflow analysis, this silently drops valid data.")

**2. Wrong groupby logic producing misleading statistics**

AI assistants sometimes group by the wrong column or forget to include a necessary grouping key, producing statistics that look reasonable but are actually aggregated incorrectly.

In [ ]:
# Demonstration: wrong groupby key
# Task: compute mean monthly discharge for EACH water year

# WRONG: grouping by month only (averages across ALL years)
wrong_monthly = df_ai.groupby("month", as_index=False)["discharge_af"].mean()
print("WRONG — grouped by month only (averages across all years):")
print(wrong_monthly.head())
print(f"  -> Only {len(wrong_monthly)} rows (one per calendar month)")
print()

# CORRECT: grouping by water_year AND month
correct_monthly = df_ai.groupby(
    ["water_year", "month"], as_index=False
)["discharge_af"].mean()
print("CORRECT — grouped by water year and month:")
print(correct_monthly.head())
print(f"  -> {len(correct_monthly)} rows (one per year-month combination)")
print()
print("The wrong version hides year-to-year variability — a critical detail")
print("for drought analysis or trend detection.")

### When to Use AI Here

- ✅ **Well-suited for AI:** Reading and parsing CSV files, setting up DatetimeIndex, boilerplate for `groupby` + `aggregate` patterns
- ✅ **Well-suited for AI:** Generating unit conversion code when you specify the exact conversion factor or formula
- ✅ **Well-suited for AI:** Writing repetitive column renaming, dropping, or type-casting operations
- ⚠️ **Requires manual understanding:** Choosing the correct groupby keys for your specific analysis question
- ⚠️ **Requires manual understanding:** Verifying index alignment when merging DataFrames from different sources
- ⚠️ **Requires manual understanding:** Interpreting whether summary statistics are physically reasonable (e.g., is a mean daily discharge of 0.01 acre-ft/day realistic for this river?)

---

### 🤖 AI Exercise Variant: Process and Clean NWIS Streamflow Data

Now it's your turn to use an AI assistant (or imagine using one) to tackle a pandas data-processing task. This exercise mirrors the manual data manipulation you practiced above, but focuses on the AI-assisted workflow.

**Your task:** Use an AI coding assistant to write code that:
1. Reads the Russian River gage data CSV (`data/pandas/RR_gage_data.csv`)
2. Filters to a single site of your choice
3. Drops rows with missing discharge values
4. Adds a water-year column
5. Converts discharge from cfs to acre-feet per day
6. Computes the **yearly total discharge** in acre-feet (hint: multiply daily mean by number of days, or use `sum` on the daily acre-ft values)
7. Identifies the **wettest and driest water years** on record

**Ground truth values for verification** (for site 11467000, Russian River near Guerneville):
- Conversion factor: 1 cfs = 1.9835 acre-ft/day (= 86400 / 43559.9)
- Water year assignment: Oct–Dec belong to the *following* year's water year
- The yearly total should be the sum of daily acre-ft values within each water year

**Step 1:** Write your prompt in the cell below. Be specific about the input file, column names, the conversion factor, and what "water year" means.

In [ ]:
# Write your prompt here as a comment or string:
my_prompt = """
YOUR PROMPT HERE
"""

**Step 2:** Paste or write the AI-generated code in the cell below, then run it.

In [ ]:
# Paste your AI-generated data processing code here:



**Step 3:** Verify your results against these checks.

In [ ]:
# Verification tests — uncomment and fill in your variable names

# Test 1: Conversion factor check
# expected_factor = 86400 / 43559.9  # 1.9835
# print(f"Conversion factor used: ___  (expected: {expected_factor:.4f})")

# Test 2: Water year check — pick a known date and verify
# e.g., December 2020 should be water year 2021

# Test 3: Yearly totals should be positive and physically reasonable
# A large river like the Russian River near Guerneville typically
# has yearly totals in the range of ~100,000 to ~2,000,000 acre-ft
# print(f"Min yearly total: ___ acre-ft")
# print(f"Max yearly total: ___ acre-ft")

# Test 4: Driest and wettest years should be identifiable
# print(f"Driest water year: ___")
# print(f"Wettest water year: ___")

### ✅ Verification Checklist

- [ ] Code runs without errors
- [ ] Output matches Ground Truth: conversion factor ≈ 1.9835, water year assignment is correct
- [ ] Domain review: yearly totals are physically reasonable for the Russian River (∼100k–2M acre-ft/yr)
- [ ] Code quality: no unnecessary complexity, uses pandas idioms (groupby, dropna), follows Python conventions
- [ ] Groupby logic: yearly totals are grouped by water year, not calendar year

### Prompt Log

In [ ]:
# Prompt Log
# ----------
# Prompt version 1: (paste your first prompt here)
#
# Result: (did the AI-generated code work? what was wrong?)
#
# Prompt version 2 (if needed): (paste your revised prompt here)
#
# Result: (did the revision fix the issue?)
#
# Number of iterations to reach correct result: 
#
# What did you learn about prompting for pandas data manipulation?